# DACE — point-wise cost baseline (evaluation)

**DACE** ("A Database-Agnostic Cost Estimator", Liang et al., ICDE 2024,
[github.com/liang-zibo/DACE](https://github.com/liang-zibo/DACE)) is a Tier-1
point estimator that *cites and builds on* the Zero-Shot Cost Model code base,
which is why it sits alongside `zero_shot` and `qppnet` here.

It flattens the plan into a **pre-order node sequence**, runs a single-head
**Transformer encoder** whose self-attention is masked by **tree reachability**
(a node attends only to itself and its descendants, so the root attends to the
whole plan), and reads out the root through a small MLP with a **sigmoid runtime
head** (runtime as a fraction of a fitted `max_runtime`).

It consumes the **same** `build_graph_dataset` tensors as QPPNet / Zero-Shot — no
new featureisation. Two lakehouse caveats (documented in `dace/model.py`):

- **Root-only supervision.** DACE normally supervises every sub-plan against its
  Postgres `Actual Total Time`. Trino/Iceberg plans carry only a whole-query
  runtime, so the DACE Q-error log loss is applied at the **root** only.
- **No LoRA.** DACE's LoRA layers exist purely for its cross-database fine-tuning
  transfer story; a single lakehouse has nothing to transfer from, so we use
  plain `nn.Linear`.

Being a point estimator, it fills only the **point-error** sheet; CRPS / interval
/ uncertainty sheets are `n/a` for Tier-1.


In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from loader.load import load_aligned_plans_and_runs

from uncertainty_prediction.src import *
from uncertainty_prediction.config import *

from uncertainty_prediction.baselines.predictive.common import (
    build_graph_dataset,
    evaluate_point_predictions,
    print_metric_headers_for_excel,
    print_metrics_for_excel,
)
from uncertainty_prediction.baselines.predictive.dace import DACEBaseline

import torch

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
queries_dir = "/mnt/lakehouse-raw-results/tpcds/lakehouse-a/20260222-191819Z/queries"

plans_by_query, runs_by_query, common = load_aligned_plans_and_runs(
    queries_dir=queries_dir,
    run_ids=RUN_IDS,
    collection=COLLECTION_NAME,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    metric=METRIC,
    xcol=XCOL,
    ycol=YCOL,
    parsed_results_root=PARSED_RESULTS_ROOT,
    canon_fn=canon_qid,
    min_runs=1,
    min_points_per_run=2,
    require_cols=(XCOL, YCOL),
)

train_qids, test_qids = split_query_ids(common, seed=SEED, test_frac=TEST_FRAC)
print("n_train:", len(train_qids), "n_test:", len(test_qids))

In [ ]:
# structured plan graphs — identical tensors QPPNet / Zero-Shot consume
gdata = build_graph_dataset(
    plans_by_query=plans_by_query, runs_by_query=runs_by_query,
    train_qids=train_qids, test_qids=test_qids, xcol=XCOL, runtime_mode="mean",
)
print("num_ops:", gdata["num_ops"], "| cont_dim:", gdata["cont_dim"],
      "| node_length:", gdata["num_ops"] + gdata["cont_dim"])

y_test_log = gdata["y_test_log"]
y_test_runtime = np.exp(y_test_log)

## Train + evaluate DACE

In [ ]:
dace = DACEBaseline(num_ops=gdata["num_ops"], cont_dim=gdata["cont_dim"], device=device, seed=42)
dace.fit(gdata["train_graphs"], gdata["y_train_log"], num_epochs=100, lr=1e-3, verbose=True)

mu_log = dace.predict_point_log(gdata["test_graphs"])
dace_metrics = evaluate_point_predictions(np.exp(mu_log), y_test_runtime)
print_metric_headers_for_excel(dace_metrics)
print_metrics_for_excel(dace_metrics)

## Point-error summary

DACE is Tier-1: only point-error columns are populated (CRPS / coverage / MPIW /
uncertainty are `n/a` by design). Paste the tab-separated rows above straight
into the workbook's point-error sheet, next to QPPNet and Zero-Shot.


In [ ]:
summary = pd.DataFrame({"DACE": dace_metrics}).T
cols = ["mae", "rmse", "median_q_error"]
summary.reindex(columns=[c for c in cols if c in dace_metrics])